# 04 · Decision engine: approve or review

**Question:** given what was read from the receipt (notebook 03), the claim as the rep entered it, the employee and the claim history,
should the claim be paid without a clerk opening it?

**Design**
* The OCR/VLM only **reads**. Every decision is a plain Python rule that traces to a section of `expense-policy.pdf` (v3.2).
* `approve` only when **every** blocking check passes; otherwise `review`, with **every** failing reason code.
* Three separate steps, as the service will call them:
  `fields = extract_receipt(photo)` → `validation = validate_fields(fields, claim, employee)` → `decide_claim(fields, validation, claim, employee, history)`.
* No ML classifier: 283 historical decisions, several of them wrong, are not a training set for "what the policy says".

**How it is judged.** Clerk decisions are *not* ground truth: the clerks approved re-claimed bills, a colleague's fuel bills and a
pen-altered total. So there are two separate evaluations: **(A) agreement with the clerks** (a workflow comparison) and
**(B) policy correctness** against a reference outcome built from independent evidence. The headline safety number is
**unsafe auto-approvals**: claims the engine would pay that the policy says a person must look at.

In [1]:
import hashlib
import inspect
import re
import sys
import time
from dataclasses import dataclass, field
from difflib import SequenceMatcher
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from tarang.fields import clean_text, find_dates, norm, similarity
from tarang.images import injection_hits
from tarang.ocr import load_outputs

read = lambda p: pd.read_csv(p, dtype=str, keep_default_na=False)
train = read(ROOT / "data" / "processed" / "claims_train_clean.csv").assign(split="train")
test = read(ROOT / "data" / "processed" / "claims_test_clean.csv").assign(split="test")
claims = pd.concat([train, test], ignore_index=True).fillna("")
employees = read(ROOT / "data" / "raw" / "employees.csv").set_index("emp_id")
fields_all = read(ROOT / "data" / "processed" / "receipt_fields.csv")
FIELDS = fields_all[fields_all.method == "combined"].set_index("claim_id")      # best reader from notebook 03
audit = read(ROOT / "reports" / "image_audit.csv").set_index("claim_id")
dup_photos = read(ROOT / "reports" / "duplicate_images.csv").set_index("claim_id")
visual = read(ROOT / "reports" / "visual_review.csv")
VLM = load_outputs(ROOT / "data" / "interim")["paddleocr_vl_retry"]
print(f"{len(train)} train + {len(test)} test claims, {len(employees)} employees")

283 train + 190 test claims, 16 employees


## 1 · The policy, as rules

`expense-policy.pdf` sections used (P2 = section 2, ...):

| § | Policy text (abridged) | Rule |
|---|---|---|
| P2 | Fuel: own two-wheeler (ASM: own car). **The bill must show the vehicle number.** | vehicle on the bill must be the claimant's |
| P2 | Food: meals with doctors up to Rs 2,500 per bill without prior approval | covered by the Rs 2,000 clerk threshold |
| P2 | Lodging: **outstation nights only**; M1/M2 up to **Rs 3,000/night before GST**, M3 Rs 4,500; **GST tax invoice in the employee's name** | four lodging checks |
| P2 | Local travel: with the driver's receipt; Parking: daily parking near clinics is **expected** | no rule against daily Rs 30 parking |
| P3 | **The claim amount is the amount on the bill. Rounding to the nearest Rs 10 is accepted.** | claimed = bill, or bill rounded |
| P3 | **A bill is claimed once, by the person who paid it.** A shared bill is claimed once, by the payer. | duplicate bill / photo; memo name |
| P3 | **A bill must not be altered.** Notes (doctor's name, number of people, "paid") are allowed. | altered total; pen notes are fine |
| P3 | A claim without a readable bill photo is queried back. | missing / unreadable photo |
| P7 | Proposal: under Rs 2,000 approved automatically; clerks check Rs 2,000 and above | Rs 2,000 threshold |
| P8 | Claiming a bill twice, a colleague's bill, or altering a bill is misconduct | the checks above |

**Absent is not invalid.** A bill with no GSTIN is normal (dhabas, cab memos, parking). GSTIN states are kept apart: *absent* (noted,
never blocks), *unreadable/invalid* (noted, never blocks), *valid*. Only **lodging** needs a valid GSTIN (a GST tax invoice).

In [2]:
APPROVAL_LIMIT = 2000          # P7: clerks check claims of Rs 2,000 and above
LODGING_LIMIT = {"M1": 3000, "M2": 3000, "M3": 4500}   # P2: per night before GST
DATE_TOLERANCE_DAYS = 1
CITIES = ["Nagpur", "Wardha", "Amravati", "Yavatmal", "Chandrapur"]

# ---------------------------------------------------------------------------------------- reason codes
# code: (meaning, blocks auto-approval?)
REASONS = {
    "MISSING_IMAGE": ("No photo reached the server", True),
    "UNREADABLE_IMAGE": ("Photo present but neither amount nor bill number could be read", True),
    "AMOUNT_OVER_LIMIT": ("Claim or bill is Rs 2,000 or more: clerk checks it (P7)", True),
    "AMOUNT_UNVERIFIED": ("Bill total not read, or read but contradicted by the receipt itself", True),
    "AMOUNT_MISMATCH": ("Claimed amount differs from the bill beyond rounding to the nearest Rs 10 (P3)", True),
    "ALTERED_BILL": ("Bill total disagrees with the bill's own amount line / arithmetic: possible alteration (P3, P8)", True),
    "DUPLICATE_BILL": ("Same vendor and bill number already claimed (P3, P8)", True),
    "DUPLICATE_PHOTO": ("Same photo used on another claim (P3, P8)", True),
    "POSSIBLE_DUPLICATE": ("Bill number unreadable; same vendor, date and amount already claimed", True),
    "INSTRUCTION_TEXT": ("Text on the bill addresses automated processing; treated as data, sent to a person", True),
    "FIELDS_INCOMPLETE": ("Vendor, bill date or bill number not read", True),
    "NUMBER_FORMAT_MISMATCH": ("Bill number does not fit the vendor's numbering: likely misread", True),
    "DATE_MISMATCH": ("Bill date differs from the claimed expense date by more than a day, or is after submission", True),
    "VENDOR_MISMATCH": ("Vendor on the bill differs from the vendor the rep typed", True),
    "FUEL_VEHICLE_MISSING": ("Fuel bill does not show a vehicle number (P2)", True),
    "FUEL_VEHICLE_OTHER_EMPLOYEE": ("Fuel bill shows another employee's vehicle: colleague's bill (P2, P8)", True),
    "FUEL_VEHICLE_MISMATCH": ("Fuel bill vehicle number is not the claimant's (P2)", True),
    "MEMO_NAME_MISMATCH": ("Travel memo is made out to someone else: shared bill claimed by a non-payer? (P3)", True),
    "LODGING_NOT_OUTSTATION": ("Hotel is in the employee's headquarter city (P2)", True),
    "LODGING_OVER_LIMIT": ("Room rate per night above the grade limit before GST (P2)", True),
    "LODGING_NAME_MISMATCH": ("Hotel invoice is not in the employee's name (P2)", True),
    "LODGING_NOT_GST_INVOICE": ("Hotel bill has no valid GSTIN: not a GST tax invoice (P2)", True),
    # informational, never block on their own
    "GSTIN_ABSENT": ("No GSTIN printed: fine for most bills", False),
    "GSTIN_UNREADABLE": ("A GSTIN is printed but could not be read with a valid checksum", False),
    "ROUNDED_TO_10": ("Claim is the bill rounded to the nearest Rs 10: accepted (P3)", False),
}

# Policy matrix: every rule, where it comes from, and the regression test that proves it.
POLICY_MATRIX = [
    ("P3", "No readable bill photo", "review", "MISSING_IMAGE / UNREADABLE_IMAGE", "test_missing_photo_goes_to_review"),
    ("P7", "Claim or bill >= Rs 2,000", "review", "AMOUNT_OVER_LIMIT", "test_over_limit_goes_to_review"),
    ("P3", "Bill total unread or self-contradicted", "review", "AMOUNT_UNVERIFIED", "test_unverified_amount_goes_to_review"),
    ("P3", "Claimed != bill, beyond nearest-Rs-10 rounding", "review", "AMOUNT_MISMATCH", "test_amount_mismatch_and_rounding"),
    ("P3/P8", "Bill total contradicts its own amount line or arithmetic", "review", "ALTERED_BILL", "test_altered_fuel_total"),
    ("P3/P8", "Same vendor + bill number claimed before (any employee)", "review", "DUPLICATE_BILL", "test_duplicate_bill_*"),
    ("P3/P8", "Same photo on another claim", "review", "DUPLICATE_PHOTO", "test_duplicate_photo_pairs"),
    ("P3/P8", "Bill no. unreadable, same vendor + date + amount claimed before", "review", "POSSIBLE_DUPLICATE", "test_possible_duplicate"),
    ("-", "Instruction-like text on the bill", "review", "INSTRUCTION_TEXT", "test_instruction_stamp"),
    ("P4", "Vendor / date / bill number not read", "review", "FIELDS_INCOMPLETE", "test_incomplete_fields"),
    ("P4", "Bill number outside the vendor's format", "review", "NUMBER_FORMAT_MISMATCH", "test_number_format"),
    ("P4", "Bill date != expense date (>1 day) or after submission", "review", "DATE_MISMATCH", "test_date_mismatch"),
    ("P4", "Bill vendor != vendor typed by the rep", "review", "VENDOR_MISMATCH", "test_vendor_mismatch"),
    ("P2", "Fuel bill without vehicle number", "review", "FUEL_VEHICLE_MISSING", "test_fuel_vehicle_rules"),
    ("P2/P8", "Fuel bill with a colleague's vehicle", "review", "FUEL_VEHICLE_OTHER_EMPLOYEE", "test_fuel_vehicle_rules"),
    ("P2", "Fuel bill with an unknown vehicle", "review", "FUEL_VEHICLE_MISMATCH", "test_fuel_vehicle_rules"),
    ("P3", "Travel memo in another person's name", "review", "MEMO_NAME_MISMATCH", "test_memo_name"),
    ("P2", "Lodging in headquarter city", "review", "LODGING_NOT_OUTSTATION", "test_lodging_rules"),
    ("P2", "Lodging rate above grade limit (before GST)", "review", "LODGING_OVER_LIMIT", "test_lodging_rules"),
    ("P2", "Hotel invoice not in employee's name", "review", "LODGING_NAME_MISMATCH", "test_lodging_rules"),
    ("P2", "Hotel bill without valid GSTIN", "review", "LODGING_NOT_GST_INVOICE", "test_lodging_rules"),
    ("P3", "Pen notes (doctor, pax, 'paid'), daily Rs 30 parking", "no effect", "-", "test_allowed_notes_and_parking"),
    ("-", "GSTIN absent / unreadable on a non-hotel bill", "no effect (noted)", "GSTIN_ABSENT / GSTIN_UNREADABLE", "test_gstin_states"),
    ("P7", "Everything above passes, claim < Rs 2,000", "approve", "-", "test_clean_claim_is_approved"),
]

In [3]:
pd.DataFrame(POLICY_MATRIX, columns=["policy", "condition", "action", "reason code", "regression test"])

,policy,condition,action,reason code,regression test
0,P3,No readable bill photo,review,MISSING_IMAGE / UNREADABLE_IMAGE,test_missing_photo_goes_to_review
1,P7,"Claim or bill >= Rs 2,000",review,AMOUNT_OVER_LIMIT,test_over_limit_goes_to_review
2,P3,Bill total unread or self-contradicted,review,AMOUNT_UNVERIFIED,test_unverified_amount_goes_to_review
3,P3,"Claimed != bill, beyond nearest-Rs-10 rounding",review,AMOUNT_MISMATCH,test_amount_mismatch_and_rounding
4,P3/P8,Bill total contradicts its own amount line or ...,review,ALTERED_BILL,test_altered_fuel_total
5,P3/P8,Same vendor + bill number claimed before (any ...,review,DUPLICATE_BILL,test_duplicate_bill_*
6,P3/P8,Same photo on another claim,review,DUPLICATE_PHOTO,test_duplicate_photo_pairs
7,P3/P8,"Bill no. unreadable, same vendor + date + amou...",review,POSSIBLE_DUPLICATE,test_possible_duplicate
8,-,Instruction-like text on the bill,review,INSTRUCTION_TEXT,test_instruction_stamp
9,P4,Vendor / date / bill number not read,review,FIELDS_INCOMPLETE,test_incomplete_fields


## 2 · Why these rules: evidence in the data

Before writing rules, each policy point was checked against the claims. Four findings shaped the engine.

In [4]:
def with_fields(df):
    out = df.join(FIELDS[["vendor", "receipt_date", "receipt_number", "total_amount", "total_status"]].add_prefix("bill_"), on="claim_id")
    out["bill_key"] = [f"{norm(v)}|{n}" if v and n else "" for v, n in zip(out.bill_vendor, out.bill_receipt_number)]
    return out

cf = with_fields(claims)
same_bill = cf[(cf.bill_key != "") & cf.duplicated("bill_key", keep=False)].sort_values(["bill_key", "submitted_on"])
pairs = same_bill.groupby("bill_key")
both_approved = sum((g.status == "approved").sum() >= 2 for _, g in pairs)
print(f"(1) Same bill claimed more than once (vendor + bill number as read): {pairs.ngroups} bills, {len(same_bill)} claims")
print(f"    photographed again, so image hashing finds only {dup_photos.duplicate_group.nunique()} of them; "
      f"in train, {both_approved} bills were approved twice by the clerks, 1 was caught (CLM-31079)")
same_bill[["bill_key", "claim_id", "split", "emp_id", "submitted_on", "bill_receipt_date", "amount_claimed", "status", "clerk_note"]].head(16)

(1) Same bill claimed more than once (vendor + bill number as read): 28 bills, 56 claims
    photographed again, so image hashing finds only 4 of them; in train, 15 bills were approved twice by the clerks, 1 was caught (CLM-31079)


,bill_key,claim_id,split,emp_id,submitted_on,bill_receipt_date,amount_claimed,status,clerk_note
6,annapurna dining hall|A/1202,CLM-30428,train,TPD-0452,2026-06-12,2026-06-12,536.00,approved,
428,annapurna dining hall|A/1202,CLM-31418,test,TPD-0452,2026-08-05,2026-06-12,536.00,,
106,annapurna dining hall|A/1324,CLM-30661,train,TPD-0503,2026-06-25,,1559.00,approved,
168,annapurna dining hall|A/1324,CLM-30804,train,TPD-0452,2026-07-03,,1559.00,approved,
141,ashirwad veg restaurant|5967,CLM-30741,train,TPD-0423,2026-06-30,2026-06-21,1102.00,approved,
236,ashirwad veg restaurant|5967,CLM-30982,train,TPD-0423,2026-07-11,2026-06-21,1102.00,approved,
107,cafe coffee corner|A/1406,CLM-30664,train,TPD-0471,2026-06-25,2026-06-21,987.00,approved,verified
454,cafe coffee corner|A/1406,CLM-31475,test,TPD-0471,2026-08-07,2026-06-21,987.00,,
10,green leaf restaurant|A/6178,CLM-30436,train,TPD-0459,2026-06-13,2026-06-13,630.00,approved,
235,green leaf restaurant|A/6178,CLM-30980,train,TPD-0463,2026-07-11,2026-06-13,630.00,approved,


In [5]:
VEH = re.compile(r"veh(?:icle)?\s*no\.?\s*[:.]?\s*([A-Z]{2}\s?\d{1,2}\s?[A-Z]{0,3}\s?\d{3,4})", re.I)
owner = {re.sub(r"\s", "", v).upper(): e for e, v in employees.vehicle_no.items()}
rows = []
for _, c in claims[(claims.category == "Fuel") & (claims.photo_available == "True")].iterrows():
    m = VEH.search(clean_text(VLM.loc[c.image_file].text))
    plate = re.sub(r"\s", "", m.group(1).upper()) if m else ""
    own = re.sub(r"\s", "", employees.loc[c.emp_id, "vehicle_no"]).upper()
    if plate and plate != own:
        rows.append({"claim_id": c.claim_id, "split": c.split, "claimant": c.emp_id, "own vehicle": own, "bill vehicle": plate,
                     "bill vehicle belongs to": owner.get(plate, "nobody (OCR slip?)"), "clerk": c.status})
print("(2) Fuel bills whose vehicle is not the claimant's:")
pd.DataFrame(rows)

(2) Fuel bills whose vehicle is not the claimant's:


,claim_id,split,claimant,own vehicle,bill vehicle,bill vehicle belongs to,clerk
0,CLM-30925,train,TPD-0459,MH34BT9042,MH34BT9842,nobody (OCR slip?),approved
1,CLM-30954,train,TPD-0412,MH31EK4471,MH31FA2290,TPD-0418,approved
2,CLM-30978,train,TPD-0412,MH31EK4471,MH31FA2290,TPD-0418,approved
3,CLM-31023,train,TPD-0452,MH32BB1205,MH32AL6618,TPD-0447,approved
4,CLM-31120,test,TPD-0431,MH31DX0981,MH31FA2290,TPD-0418,
5,CLM-31228,test,TPD-0431,MH31DX0981,MH31DX8981,nobody (OCR slip?),
6,CLM-31386,test,TPD-0418,MH31FA2290,MH31FA2298,nobody (OCR slip?),


Two kinds: **a colleague's vehicle exactly** (CLM-30954, CLM-30978, CLM-31023 in train, all checked on the photo and all approved by
the clerks; CLM-31120 in test), and **one-character OCR slips** of the claimant's own plate (`MH34BT9842` vs `9042`). The rule allows
one character of difference from the claimant's own plate and flags an exact match with another employee's plate.

In [6]:
t = with_fields(train)
t = t[t.bill_receipt_date != ""]
gap = (pd.to_datetime(t.expense_date) - pd.to_datetime(t.bill_receipt_date)).dt.days
print("(3) expense date typed by the rep minus bill date (days):", gap.value_counts().sort_index().to_dict())
far = t[gap.abs() > 1]
far.assign(gap_days=gap[gap.abs() > 1], is_reclaim=far.bill_key.isin(same_bill.bill_key))[
    ["claim_id", "expense_date", "bill_receipt_date", "gap_days", "is_reclaim", "status", "clerk_note"]]

(3) expense date typed by the rep minus bill date (days): {-5: 1, 0: 241, 1: 1, 2: 1, 4: 1, 7: 2, 8: 1, 9: 1, 10: 1, 14: 1}


,claim_id,expense_date,bill_receipt_date,gap_days,is_reclaim,status,clerk_note
22,CLM-30469,2026-06-15,2026-06-05,10,True,approved,GST bill attached
108,CLM-30665,2026-06-19,2026-06-05,14,True,approved,
119,CLM-30689,2026-06-17,2026-06-13,4,True,approved,
182,CLM-30840,2026-06-21,2026-06-14,7,True,approved,
229,CLM-30963,2026-06-14,2026-06-19,-5,True,approved,
233,CLM-30978,2026-06-21,2026-06-14,7,False,approved,
240,CLM-30991,2026-06-24,2026-06-15,9,True,approved,
250,CLM-31015,2026-07-09,2026-07-07,2,False,approved,GST invoice ok
280,CLM-31079,2026-06-30,2026-06-22,8,True,rejected,same bill as CLM-30679


Almost every claim dated more than a day away from its bill is a **re-claim of an old bill under a new date**, so a date mismatch is a
useful duplicate signal on its own. (CLM-31015 is a hotel stay; for lodging any night of the stay is accepted.)

**(4) Altered totals.** Six bills carry a digit added in front of the printed total by pen, all checked on the photo. In train:
CLM-30919 (300 → 800), CLM-31009 (536.01 → 1536.01, the one the clerks rejected), CLM-30568 (514 → 1514) and CLM-30764 (268 → 1268);
in test CLM-31299 (472 → 1472) and CLM-31335 (313.77 → 1313.77). Three of the four in train were approved at the altered amount.
The receipt's own arithmetic exposes them: fuel slips print rate, volume and an amount line; restaurant bills print sub total and taxes.
`fields.parse_total` marks such a total `inconsistent` and reports the self-consistent amount. When the VLM doesn't read the added
digit, the claim still fails on **claimed ≠ bill**.

**(5) Cab memo in someone else's name.** CLM-31497 (test): Sagar claims a memo made out to Prachi, who claimed the same memo herself
(CLM-31310): a shared bill claimed by a non-payer.

## 3 · The engine

**Extra details** the five fields don't cover: vehicle number, hotel city, guest name, nights and room rate, the memo's name, and any
instruction-like text.

In [7]:
# ---------------------------------------------------------------------------------------- extra receipt details
VEHICLE = re.compile(r"veh(?:icle)?\s*no\.?\s*[:.]?\s*([A-Z]{2}\s?\d{1,2}\s?[A-Z]{0,3}\s?\d{3,4})", re.I)
NUM = r"(\d{1,3}(?:,\d{3})+(?:\.\d{1,2})?|\d+(?:\.\d{1,2})?)"


def _after(label, text, same_line_only=False):
    """Number after a label, on the same line or (VLM layout) the next line."""
    lines = text.splitlines()
    for i, l in enumerate(lines):
        if re.search(label, l, re.I):
            m = re.search(label + r"[^\d]*" + NUM, l, re.I)
            if m:
                return float(m.group(m.lastindex).replace(",", ""))
            if not same_line_only and i + 1 < len(lines):
                n = re.fullmatch(r"\s*(?:rs\.?)?\s*" + NUM + r"\s*", lines[i + 1], re.I)
                if n:
                    return float(n.group(1).replace(",", ""))
    return None


def parse_details(text):
    """Policy-relevant details beyond the five fields: vehicle, fuel arithmetic, hotel stay, memo name, instruction text."""
    t = clean_text(text)
    d = {}
    m = VEHICLE.search(t)
    d["vehicle_number"] = re.sub(r"\s", "", m.group(1).upper()) if m else ""
    d["fuel_rate"] = _after(r"rate\s*\(rs/l\)", t)
    d["fuel_volume"] = _after(r"volume\s*\(l\)", t)
    d["fuel_amount_line"] = _after(r"amount\s*\(rs\)", t)
    d["printed_total"] = _after(r"\btotal\s*rs\.?", t)
    g = re.search(r"guest\s*:?\s*(?:mr\.?\s*/?\s*ms\.?\s*)?([A-Za-z][A-Za-z .]+)", t, re.I)
    d["guest_name"] = g.group(1).strip() if g else ""
    d["hotel_city"] = next((c for l in t.splitlines()[:4] for c in CITIES if c.lower() in l.lower()), "")
    d["nights"] = _after(r"\bnights\b", t, same_line_only=True)
    room = re.search(r"room\s*tariff.*?\n?.*?([\d,]+\.\d\d)", t, re.I | re.S)
    d["room_rate"] = float(room.group(1).replace(",", "")) if room else None
    nm = re.search(r"\bname\s*:?\s*([A-Za-zÀ-ſ]+)", t, re.I)
    d["memo_name"] = nm.group(1) if nm else ""
    d["instruction_hits"] = "; ".join(injection_hits(t))
    return d

**Field validation and policy checks** (no history needed).

In [8]:
# ---------------------------------------------------------------------------------------- validation
@dataclass
class Check:
    name: str
    passed: bool
    reason: str = ""          # reason code when the check fails (or an informational code)
    detail: str = ""


def _amount(s):
    try:
        return float(s)
    except (TypeError, ValueError):
        return None


def rounding_ok(claimed, bill):
    """P3: the claim is the bill amount, or the bill rounded to the nearest Rs 10 (rounding to the rupee is gentler, also fine)."""
    if claimed is None or bill is None:
        return False
    allowed = {round(bill, 2), float(int(bill + 0.5)), float(int(bill / 10 + 0.5) * 10)}
    return any(abs(claimed - a) < 0.005 for a in allowed)


def plate_distance(a, b):
    return sum(x != y for x, y in zip(a, b)) + abs(len(a) - len(b))


def validate_fields(fields, claim, employee, details=None, employees=None, vendor_shapes=None):
    """Field-level and policy checks for one claim. Returns a list of Check (no history needed here)."""
    details = details or {}
    checks = []
    if claim.get("photo_available") not in (True, "True"):
        return [Check("photo", False, "MISSING_IMAGE", "photo never reached the server")]

    claimed = _amount(claim.get("amount_claimed"))
    bill = _amount(fields.get("total_amount"))
    readable = bool(fields.get("total_amount") or fields.get("receipt_number"))
    checks.append(Check("readable", readable, "UNREADABLE_IMAGE", "no amount and no bill number read"))
    if not readable:
        return checks

    limit_hit = max(x for x in (claimed, bill, 0) if x is not None) >= APPROVAL_LIMIT
    checks.append(Check("under_limit", not limit_hit, "AMOUNT_OVER_LIMIT", f"claimed {claimed}, bill {bill}"))

    total_status = fields.get("total_status", "").replace("_fallback", "")
    # altered bill: the printed total contradicts the bill's own arithmetic (fuel rate x volume and amount line, or
    # sub total + taxes); fields.parse_total then reports the self-consistent amount
    altered = total_status == "inconsistent"
    checks.append(Check("not_altered", not altered, "ALTERED_BILL", f"printed total contradicts the bill's own arithmetic; consistent amount {bill}"))
    amount_ok = total_status in ("verified", "ok", "words_only", "inconsistent") and not _truthy(fields.get("looped"))
    checks.append(Check("amount_read", amount_ok and bill is not None, "AMOUNT_UNVERIFIED", f"status {fields.get('total_status')}"))

    if bill is not None and claimed is not None:
        match = rounding_ok(claimed, bill)
        checks.append(Check("amount_matches_claim", match, "AMOUNT_MISMATCH", f"claimed {claimed} vs bill {bill}"))
        if match and abs(claimed - bill) >= 0.005:
            checks.append(Check("rounding", True, "ROUNDED_TO_10", f"{bill} -> {claimed}"))

    missing = [f for f, st in (("vendor", "vendor_status"), ("receipt_date", "date_status"), ("receipt_number", "number_status"))
               if not fields.get(f) or fields.get(st) in ("absent", "unknown")]
    checks.append(Check("fields_complete", not missing, "FIELDS_INCOMPLETE", "missing: " + ", ".join(missing)))
    checks.append(Check("number_format", fields.get("number_status") != "format_mismatch", "NUMBER_FORMAT_MISMATCH", fields.get("receipt_number", "")))

    if fields.get("receipt_date"):
        bd = pd.Timestamp(fields["receipt_date"])
        ed = pd.Timestamp(claim["expense_date"]) if claim.get("expense_date") else None
        late = claim.get("submitted_on") and bd > pd.Timestamp(claim["submitted_on"])
        far = ed is not None and abs((ed - bd).days) > DATE_TOLERANCE_DAYS
        if claim.get("category") == "Lodging" and details.get("nights"):
            far = ed is not None and not (bd - pd.Timedelta(days=int(details["nights"])) <= ed <= bd)   # any night of the stay
        checks.append(Check("date_matches_claim", not (late or far), "DATE_MISMATCH", f"bill {fields['receipt_date']} vs expense {claim.get('expense_date')}"))

    if fields.get("vendor") and claim.get("vendor"):
        checks.append(Check("vendor_matches_claim", similarity(fields["vendor"], claim["vendor"]) >= 0.6, "VENDOR_MISMATCH",
                            f"bill {fields['vendor']} vs typed {claim['vendor']}"))

    gst_state = fields.get("gstin_status", "absent").replace("_fallback", "")
    if gst_state == "absent":
        checks.append(Check("gstin", True, "GSTIN_ABSENT"))
    elif gst_state == "invalid":
        checks.append(Check("gstin", True, "GSTIN_UNREADABLE", fields.get("gstin_read", "")))

    if details.get("instruction_hits") or _truthy(claim.get("instruction_stamp")):
        checks.append(Check("no_instruction_text", False, "INSTRUCTION_TEXT", details.get("instruction_hits") or "violet stamp (image audit)"))

    category = claim.get("category")
    own = re.sub(r"\s", "", str(employee.get("vehicle_no", ""))).upper()
    if category == "Fuel":
        plate = details.get("vehicle_number", "")
        others = {} if employees is None else {re.sub(r"\s", "", v).upper(): e for e, v in employees.vehicle_no.items() if e != claim.get("emp_id")}
        if not plate:
            checks.append(Check("fuel_vehicle", False, "FUEL_VEHICLE_MISSING"))
        elif plate in others:
            checks.append(Check("fuel_vehicle", False, "FUEL_VEHICLE_OTHER_EMPLOYEE", f"{plate} belongs to {others[plate]}"))
        elif plate_distance(plate, own) > 1:
            checks.append(Check("fuel_vehicle", False, "FUEL_VEHICLE_MISMATCH", f"{plate} vs own {own}"))
        else:
            checks.append(Check("fuel_vehicle", True, "", f"{plate} ~ own {own}"))

    if category == "Local travel" and details.get("memo_name"):
        first = str(employee.get("name", "")).split()[0] if employee.get("name") else ""
        ok = SequenceMatcher(None, details["memo_name"].lower(), first.lower()).ratio() >= 0.6
        checks.append(Check("memo_name", ok, "MEMO_NAME_MISMATCH", f"memo {details['memo_name']} vs {first}"))

    if category == "Lodging":
        hq = employee.get("headquarter", "")
        city = details.get("hotel_city", "")
        checks.append(Check("lodging_outstation", not city or city != hq, "LODGING_NOT_OUTSTATION", f"hotel {city or '?'} vs HQ {hq}"))
        rate, limit = details.get("room_rate"), LODGING_LIMIT.get(employee.get("grade", ""), 3000)
        checks.append(Check("lodging_rate", rate is None or rate <= limit, "LODGING_OVER_LIMIT", f"{rate} per night vs {limit}"))
        guest = details.get("guest_name", "")
        checks.append(Check("lodging_name", not guest or similarity(guest, employee.get("name", "")) >= 0.7, "LODGING_NAME_MISMATCH",
                            f"guest {guest} vs {employee.get('name')}"))
        checks.append(Check("lodging_gst_invoice", gst_state.startswith("ok"), "LODGING_NOT_GST_INVOICE", gst_state))
    return checks


def _truthy(v):
    return v in (True, "True", "true", 1)

**History checks and the decision.** A duplicate is checked three ways: same vendor + bill number (catches a bill photographed
again), same photo (image hash from notebook 02), and, when the bill number couldn't be read, same vendor + date + amount.
History has `approved` statuses only; **approved is treated as a proxy for paid**. There is no payment record in the data.

In [9]:
# ---------------------------------------------------------------------------------------- history + decision
def bill_key(vendor, number):
    return f"{norm(vendor)}|{number.strip().upper()}" if vendor and number else ""


def history_checks(claim_id, fields, claim, history):
    """Duplicate checks against every other claim (past and same batch).

    history: DataFrame of other claims with claim_id, vendor (as read), receipt_number, receipt_date, total_amount,
    photo_group (same-photo group id or ''), status ('approved' is used as a proxy for 'paid'; there is no payment record).
    """
    others = history[history.claim_id != claim_id]
    checks = []
    key = bill_key(fields.get("vendor", ""), fields.get("receipt_number", ""))
    same_bill = others[(others.bill_key == key)] if key else others.iloc[0:0]
    if len(same_bill):
        paid = same_bill[same_bill.status == "approved"]
        checks.append(Check("not_duplicate_bill", False, "DUPLICATE_BILL",
                            f"same bill as {', '.join(same_bill.claim_id)}" + (" (approved earlier: likely already paid)" if len(paid) else "")))
    group = claim.get("photo_group", "")
    if group:
        twins = others[others.photo_group == group]
        if len(twins):
            checks.append(Check("not_duplicate_photo", False, "DUPLICATE_PHOTO", f"same photo as {', '.join(twins.claim_id)}"))
    if not key and fields.get("vendor") and fields.get("receipt_date") and fields.get("total_amount"):
        twins = others[(others.vendor_key == norm(fields["vendor"])) & (others.receipt_date == fields["receipt_date"])
                       & (others.total_amount == fields["total_amount"])]
        if len(twins):
            checks.append(Check("not_possible_duplicate", False, "POSSIBLE_DUPLICATE", f"same vendor/date/amount as {', '.join(twins.claim_id)}"))
    return checks


@dataclass
class Decision:
    decision: str
    reason_codes: list = field(default_factory=list)
    notes: list = field(default_factory=list)
    reason: str = ""
    checks: list = field(default_factory=list)


def decide_claim(fields, validation, claim, employee=None, history=None):
    """approve only if every blocking check passes; otherwise review, with every failing reason code."""
    checks = list(validation) + (history_checks(claim["claim_id"], fields, claim, history) if history is not None else [])
    blocking = [c for c in checks if not c.passed and REASONS.get(c.reason, ("", True))[1]]
    notes = sorted({c.reason for c in checks if c.reason and not REASONS.get(c.reason, ("", True))[1]})   # informational codes only
    codes = list(dict.fromkeys(c.reason for c in blocking))
    if blocking:
        reason = "; ".join(f"{c.reason}: {c.detail}" if c.detail else c.reason for c in blocking)
        return Decision("review", codes, notes, reason, checks)
    return Decision("approve", [], notes, "all checks passed" + (f" ({', '.join(notes)})" if notes else ""), checks)

## 4 · Freeze the rules

The rules above were designed with the training data in view (the duplicate, vehicle and alteration cases came from it). From here
on they are **frozen**: the fingerprint below is printed into every report, and nothing is tuned on the results that follow.

In [10]:
ENGINE_SOURCE = "\n".join(inspect.getsource(f) for f in [parse_details, validate_fields, history_checks, decide_claim, rounding_ok])
RULES_VERSION = hashlib.sha256((ENGINE_SOURCE + repr(REASONS) + repr(LODGING_LIMIT) + str(APPROVAL_LIMIT)).encode()).hexdigest()[:12]
print("rules version:", RULES_VERSION)

rules version: 95bef2d00f35


## 5 · Run it

* **Train claims** are processed as they would have been live: each is checked against the claims submitted **before** it. The first
  claim of a duplicate pair was legitimate when it arrived; the second is the problem.
* **Test claims** are checked against all of train **and** every other claim in the same batch, so two copies arriving together can't
  both be paid.

In [11]:
claims["photo_group"] = claims.claim_id.map(dup_photos.duplicate_group).fillna("")
claims["instruction_stamp"] = claims.claim_id.map(audit.instruction_text).fillna("")
history = pd.DataFrame({
    "claim_id": claims.claim_id, "split": claims.split, "status": claims.status, "photo_group": claims.photo_group,
    "vendor_key": claims.claim_id.map(FIELDS.vendor).map(norm),
    "bill_key": [bill_key(FIELDS.loc[c, "vendor"], FIELDS.loc[c, "receipt_number"]) for c in claims.claim_id],
    "receipt_date": claims.claim_id.map(FIELDS.receipt_date), "total_amount": claims.claim_id.map(FIELDS.total_amount),
    "order": list(zip(claims.submitted_on, claims.claim_id)),
})


def run_claim(c):
    t0 = time.perf_counter()
    f = FIELDS.loc[c.claim_id].to_dict()
    details = parse_details(VLM.loc[c.image_file].text) if c.image_file in VLM.index else {}
    employee = employees.loc[c.emp_id].to_dict()
    validation = validate_fields(f, c.to_dict(), employee, details, employees)
    hist = history[(history.split == "train") & (history.order < (c.submitted_on, c.claim_id))] if c.split == "train" else history
    d = decide_claim(f, validation, c.to_dict(), employee, hist)
    return {"claim_id": c.claim_id, "split": c.split, "decision": d.decision, "reason_codes": "; ".join(d.reason_codes),
            "notes": "; ".join(d.notes), "reason": d.reason, "decision_ms": (time.perf_counter() - t0) * 1000}


decisions = pd.DataFrame([run_claim(c) for _, c in claims.iterrows()]).set_index("claim_id")
for s in ["train", "test"]:
    d = decisions[decisions.split == s]
    print(f"{s}: {len(d)} claims | approve {(d.decision == 'approve').mean():.1%} | review {(d.decision == 'review').mean():.1%} | "
          f"median {d.decision_ms.median():.1f} ms per claim (decision only)")
decisions.reason_codes.str.split("; ").explode().loc[lambda s: s != ""].groupby(decisions.split).value_counts().unstack(0).fillna(0).astype(int)

train: 283 claims | approve 64.7% | review 35.3% | median 0.8 ms per claim (decision only)
test: 190 claims | approve 67.9% | review 32.1% | median 0.7 ms per claim (decision only)


split,test,train
reason_codes,,
ALTERED_BILL,3,3
AMOUNT_MISMATCH,10,14
AMOUNT_OVER_LIMIT,16,27
AMOUNT_UNVERIFIED,14,22
DATE_MISMATCH,10,9
DUPLICATE_BILL,15,16
DUPLICATE_PHOTO,3,2
FIELDS_INCOMPLETE,11,32
FUEL_VEHICLE_MISSING,1,0


## 6 · Known cases (regression)

Every case found by hand in notebooks 02–04 must go to review, for the right reason.

In [12]:
KNOWN = [
    ("CLM-30919", "pen-altered total 300 -> 800, plus instruction stamp", {"AMOUNT_MISMATCH", "ALTERED_BILL"}),
    ("CLM-31009", "pen-altered total 536.01 -> 1536.01", {"ALTERED_BILL"}),
    ("CLM-30568", "pen-altered total 514 -> 1514, approved", {"ALTERED_BILL"}),
    ("CLM-30764", "pen-altered total 268 -> 1268, approved", {"ALTERED_BILL"}),
    ("CLM-31299", "pen-altered total 472 -> 1472 (test)", {"ALTERED_BILL", "AMOUNT_MISMATCH"}),
    ("CLM-31335", "pen-altered total 313.77 -> 1313.77 (test)", {"AMOUNT_MISMATCH", "ALTERED_BILL"}),
    ("CLM-30963", "same Saibaba photo as CLM-30600, both approved", {"DUPLICATE_BILL", "DUPLICATE_PHOTO"}),
    ("CLM-30804", "same Annapurna photo as CLM-30661, both approved", {"DUPLICATE_BILL", "DUPLICATE_PHOTO"}),
    ("CLM-31475", "test: same photo as approved CLM-30664", {"DUPLICATE_BILL", "DUPLICATE_PHOTO"}),
    ("CLM-31301", "test: same photo as CLM-31480, same batch", {"DUPLICATE_BILL", "DUPLICATE_PHOTO"}),
    ("CLM-31480", "test: same photo as CLM-31301, same batch", {"DUPLICATE_BILL", "DUPLICATE_PHOTO"}),
    ("CLM-30736", "Hotel Shubham invoice re-photographed, claimed 2352 then 2400", {"DUPLICATE_BILL"}),
    ("CLM-31079", "re-claimed courier bill (the one the clerks caught)", {"DUPLICATE_BILL"}),
    ("CLM-31190", "test: photo missing", {"MISSING_IMAGE"}),
    ("CLM-31280", "test: photo missing", {"MISSING_IMAGE"}),
    ("CLM-31363", "test: photo missing", {"MISSING_IMAGE"}),
    ("CLM-31391", "test: blurred beyond reading", {"UNREADABLE_IMAGE", "AMOUNT_UNVERIFIED"}),
    ("CLM-31333", "test: cropped to the header", {"UNREADABLE_IMAGE", "AMOUNT_UNVERIFIED"}),
    ("CLM-31176", "test: instruction stamp (and re-claim)", {"INSTRUCTION_TEXT"}),
    ("CLM-31401", "test: instruction stamp", {"INSTRUCTION_TEXT"}),
    ("CLM-30623", "instruction stamp, Rs 1,940 just under the limit", {"INSTRUCTION_TEXT"}),
    ("CLM-30954", "colleague's vehicle MH31FA2290", {"FUEL_VEHICLE_OTHER_EMPLOYEE"}),
    ("CLM-31023", "colleague's vehicle MH32AL6618", {"FUEL_VEHICLE_OTHER_EMPLOYEE"}),
    ("CLM-31120", "test: colleague's vehicle, bill already claimed by its owner", {"FUEL_VEHICLE_OTHER_EMPLOYEE", "DUPLICATE_BILL"}),
    ("CLM-31497", "test: cab memo in Prachi's name claimed by Sagar", {"MEMO_NAME_MISMATCH"}),
    ("CLM-30435", "claimed 568 on a 368 bill, approved", {"AMOUNT_MISMATCH"}),
]
reg = pd.DataFrame([{"claim_id": c, "case": why, "decision": decisions.loc[c, "decision"],
                     "expected reason found": bool(expect & set(decisions.loc[c, "reason_codes"].split("; "))),
                     "reason codes": decisions.loc[c, "reason_codes"]} for c, why, expect in KNOWN])
assert (reg.decision == "review").all() and reg["expected reason found"].all(), reg[(reg.decision != "review") | ~reg["expected reason found"]]
print(f"all {len(reg)} known cases -> review, with the expected reason")
reg

all 26 known cases -> review, with the expected reason


,claim_id,case,decision,expected reason found,reason codes
0,CLM-30919,"pen-altered total 300 -> 800, plus instruction...",review,True,AMOUNT_MISMATCH; INSTRUCTION_TEXT
1,CLM-31009,pen-altered total 536.01 -> 1536.01,review,True,ALTERED_BILL; AMOUNT_MISMATCH
2,CLM-30568,"pen-altered total 514 -> 1514, approved",review,True,ALTERED_BILL; AMOUNT_MISMATCH
3,CLM-30764,"pen-altered total 268 -> 1268, approved",review,True,ALTERED_BILL; AMOUNT_MISMATCH; FIELDS_INCOMPLETE
4,CLM-31299,pen-altered total 472 -> 1472 (test),review,True,ALTERED_BILL; AMOUNT_MISMATCH
5,CLM-31335,pen-altered total 313.77 -> 1313.77 (test),review,True,AMOUNT_MISMATCH
6,CLM-30963,"same Saibaba photo as CLM-30600, both approved",review,True,DATE_MISMATCH; DUPLICATE_BILL; DUPLICATE_PHOTO
7,CLM-30804,"same Annapurna photo as CLM-30661, both approved",review,True,FIELDS_INCOMPLETE; DUPLICATE_BILL; DUPLICATE_P...
8,CLM-31475,test: same photo as approved CLM-30664,review,True,DATE_MISMATCH; DUPLICATE_BILL; DUPLICATE_PHOTO
9,CLM-31301,"test: same photo as CLM-31480, same batch",review,True,DUPLICATE_BILL; DUPLICATE_PHOTO


## 7 · Evaluation A: agreement with the clerks (train)

A workflow comparison, **not** proof of correctness. Clerk `approved` ↔ engine `approve`; clerk `queried`/`rejected` ↔ engine `review`.

In [13]:
tr = decisions[decisions.split == "train"].join(train.set_index("claim_id")[["status", "amount_claimed", "clerk_note"]])
tr["clerk"] = tr.status.map({"approved": "approve", "queried": "review", "rejected": "review"})
agree = (tr.decision == tr.clerk).mean()
print(f"agreement with the clerk's decision: {agree:.1%} of {len(tr)} train claims")
pd.crosstab(tr.clerk.rename("clerk"), tr.decision.rename("engine"), margins=True)

agreement with the clerk's decision: 67.5% of 283 train claims


engine,approve,review,All
clerk,,,
approve,183,92,275
review,0,8,8
All,183,100,283


In [14]:
print("Clerk queried/rejected, engine approved (should be none):")
print(tr[(tr.clerk == "review") & (tr.decision == "approve")][["status", "clerk_note"]].to_string() or "  none")
print("\nWhy the engine sends clerk-approved claims to review:")
tr[(tr.clerk == "approve") & (tr.decision == "review")].reason_codes.str.split("; ").explode().value_counts()

Clerk queried/rejected, engine approved (should be none):
Empty DataFrame
Columns: [status, clerk_note]
Index: []

Why the engine sends clerk-approved claims to review:


reason_codes
FIELDS_INCOMPLETE              31
AMOUNT_OVER_LIMIT              27
AMOUNT_UNVERIFIED              21
DUPLICATE_BILL                 15
AMOUNT_MISMATCH                13
DATE_MISMATCH                   8
LODGING_NOT_GST_INVOICE         4
INSTRUCTION_TEXT                3
FUEL_VEHICLE_OTHER_EMPLOYEE     3
ALTERED_BILL                    2
DUPLICATE_PHOTO                 2
NUMBER_FORMAT_MISMATCH          2
LODGING_NAME_MISMATCH           1
POSSIBLE_DUPLICATE              1
Name: count, dtype: int64

Most disagreement is by design: the clerks approve Rs 2,000+ claims after checking them (the engine always sends those to a person),
and they approved re-claims and over-claims the engine flags. Agreement with the clerks rewards copying their mistakes, so it is
reported, not optimised.

## 8 · Evaluation B: policy correctness

A **reference outcome** for every train claim, built from evidence that does **not** come from the engine's own reading:
* fields as **typed by the clerk**, with the photo-checked corrections from notebook 03 (`reports/holdout_adjudication.csv`) and
  eight more found while building this notebook (listed below; the before/after effect is shown);
* missing photos; photos judged unreadable and instruction stamps found by eye (`reports/visual_review.csv`);
* duplicates by the clerk-typed vendor + bill number and by same photo, against **earlier** claims;
* alterations and colleague vehicles **checked on the photo** (listed below).

The policy then says review if any of these apply: no readable photo, Rs 2,000 or more, claimed ≠ bill beyond rounding, a re-claimed
bill, an altered bill, a colleague's vehicle, an instruction stamp, or a bill date more than a day from the claimed date.
Otherwise approve.

**What this does and doesn't show.** The rules were written knowing these train cases, so B mainly measures whether **reading errors**
make the engine approve something the evidence says it shouldn't, and how many clean claims it sends to a person needlessly.
It is not an independent test of the rules themselves.

In [15]:
from tarang.fields import canonical_vendors
CANON = canonical_vendors(train[train.clerk_vendor != ""].clerk_vendor.tolist())
adj = read(ROOT / "reports" / "holdout_adjudication.csv")
ALTERED_ON_PHOTO = {"CLM-30919", "CLM-31009", "CLM-30568", "CLM-30764"}   # pen-added digit, checked on the photo (notebooks 03-04)
COLLEAGUE_VEHICLE_ON_PHOTO = {"CLM-30954", "CLM-30978", "CLM-31023"}  # checked on the photo (this notebook)
unreadable = set(visual[visual.label == "unreadable"].claim_id)
stamped = set(visual[visual.label == "instruction_text"].claim_id)

# Clerk typing errors found on the photo while building this notebook (dev claims, outside the OCR holdout)
PHOTO_CHECKED = [
    ("CLM-30421", "clerk_bill_amount", "250.00", "TOTAL 250.00 (circled); clerk typed 240"),
    ("CLM-30543", "clerk_bill_date", "2026-06-09", "ticket 09/06/26; clerk typed 2026-09-06"),
    ("CLM-30436", "clerk_bill_no", "A/6178", "Bill No: A/6178; clerk typed A/618"),
    ("CLM-30519", "clerk_bill_date", "2026-06-14", "Date: 14.06.2026; clerk typed the 13th"),
    ("CLM-30600", "clerk_bill_date", "2026-06-19", "Date: 19/06/2026; clerk typed the 18th"),
    ("CLM-30717", "clerk_bill_no", "004023", "Receipt No: 004023; clerk typed 00402"),
    ("CLM-30568", "clerk_bill_amount", "514.00", "printed 514.00, blue pen '1' added in front; clerk typed 1514"),
    ("CLM-30764", "clerk_bill_amount", "268.00", "printed 268.00, blue pen '1' added in front; clerk typed 1268"),
]
ref_raw = train.set_index("claim_id").copy()          # clerk labels as typed, kept for the before/after comparison
ref = ref_raw.copy()
for cid, col, value, _ in PHOTO_CHECKED:
    ref.loc[cid, col] = value
for r in adj.itertuples():
    col = {"total_amount": "clerk_bill_amount", "receipt_number": "clerk_bill_no", "receipt_date": "clerk_bill_date", "gstin": "clerk_gstin"}.get(r.field)
    if col and r.claim_id in ref.index:
        ref.loc[r.claim_id, col] = r.printed_value
ref["ref_key"] = [f"{norm(CANON.get(v, v))}|{n.upper()}" if v and n else "" for v, n in zip(ref.clerk_vendor, ref.clerk_bill_no)]
ref = ref.sort_values(["submitted_on"]).assign(order=lambda d: list(zip(d.submitted_on, d.index)))


def reference_reasons(cid, r):
    out = []
    if r.photo_available != "True":
        return ["MISSING_IMAGE"]
    if cid in unreadable or not r.clerk_bill_amount:
        return ["UNREADABLE_IMAGE"]
    claimed, bill = float(r.amount_claimed), float(r.clerk_bill_amount)
    if max(claimed, bill) >= APPROVAL_LIMIT: out.append("AMOUNT_OVER_LIMIT")
    if not rounding_ok(claimed, bill): out.append("AMOUNT_MISMATCH")
    earlier = ref[ref.order < r.order]
    if r.ref_key and (earlier.ref_key == r.ref_key).any(): out.append("DUPLICATE_BILL")
    if r.name in dup_photos.index:
        twins = dup_photos[(dup_photos.duplicate_group == dup_photos.loc[r.name, "duplicate_group"])].index
        if any(t in earlier.index for t in twins): out.append("DUPLICATE_PHOTO")
    if cid in ALTERED_ON_PHOTO: out.append("ALTERED_BILL")
    if cid in COLLEAGUE_VEHICLE_ON_PHOTO: out.append("FUEL_VEHICLE_OTHER_EMPLOYEE")
    if cid in stamped: out.append("INSTRUCTION_TEXT")
    if r.clerk_bill_date and abs((pd.Timestamp(r.expense_date) - pd.Timestamp(r.clerk_bill_date)).days) > 1 and r.category != "Lodging":
        out.append("DATE_MISMATCH")
    return out


ref["ref_reasons"] = [reference_reasons(cid, r) for cid, r in ref.iterrows()]
pd.DataFrame(PHOTO_CHECKED, columns=["claim_id", "clerk column", "printed value", "evidence"])
ref["ref_decision"] = ref.ref_reasons.map(lambda x: "review" if x else "approve")
ev = tr.join(ref[["ref_decision", "ref_reasons"]])
print(f"policy reference: {(ev.ref_decision == 'approve').mean():.1%} of train claims are clean enough to auto-approve")
pd.crosstab(ev.ref_decision.rename("policy reference"), ev.decision.rename("engine"), margins=True)

policy reference: 78.8% of train claims are clean enough to auto-approve


engine,approve,review,All
policy reference,,,
approve,183,40,223
review,0,60,60
All,183,100,283


In [16]:
# the same reference built from the clerk's labels exactly as typed, to show what the photo checks changed
_ref, ref = ref, ref_raw.assign(ref_key=[f"{norm(CANON.get(v, v))}|{n.upper()}" if v and n else "" for v, n in zip(ref_raw.clerk_vendor, ref_raw.clerk_bill_no)])
ref = ref.sort_values(["submitted_on"]).assign(order=lambda d: list(zip(d.submitted_on, d.index)))
raw_reasons = pd.Series({cid: reference_reasons(cid, r) for cid, r in ref.iterrows()})
ref = _ref
raw_unsafe = ev[(ev.decision == "approve") & raw_reasons.reindex(ev.index).map(bool)]
print(f"against the clerk labels as typed: {len(raw_unsafe)} apparent unsafe approvals {list(raw_unsafe.index)}")

unsafe = ev[(ev.decision == "approve") & (ev.ref_decision == "review")]
extra = ev[(ev.decision == "review") & (ev.ref_decision == "approve")]
n_review_worthy = (ev.ref_decision == "review").sum()
print(f"UNSAFE auto-approvals: {len(unsafe)} of {n_review_worthy} review-worthy claims "
      f"({len(unsafe) / n_review_worthy:.1%}); of all claims: {len(unsafe) / len(ev):.1%}")
unsafe.assign(ref_reasons=unsafe.ref_reasons.map("; ".join))[["amount_claimed", "status", "ref_reasons", "reason"]]

against the clerk labels as typed: 4 apparent unsafe approvals ['CLM-30421', 'CLM-30543', 'CLM-30820', 'CLM-30950']
UNSAFE auto-approvals: 0 of 60 review-worthy claims (0.0%); of all claims: 0.0%


,amount_claimed,status,ref_reasons,reason
claim_id,,,,


In [17]:
print(f"Extra reviews (clean by the reference, sent to a person anyway): {len(extra)} ({len(extra) / len(ev):.1%} of claims). Why:")
extra.reason_codes.str.split("; ").explode().value_counts()

Extra reviews (clean by the reference, sent to a person anyway): 40 (14.1% of claims). Why:


reason_codes
FIELDS_INCOMPLETE         24
AMOUNT_UNVERIFIED         20
AMOUNT_OVER_LIMIT          1
AMOUNT_MISMATCH            1
NUMBER_FORMAT_MISMATCH     1
Name: count, dtype: int64

Extra reviews cost clerk time, not money: they are mostly fields the reader could not confirm (a handwritten date it skipped, a total
it couldn't cross-check). That is the price of the conservative rule "an unconfirmed field never auto-approves".

**Exception detection**: of the claims the reference flags for each reason, how many the engine sends to review:

In [18]:
det = []
for code_ in ["MISSING_IMAGE", "UNREADABLE_IMAGE", "DUPLICATE_BILL", "DUPLICATE_PHOTO", "ALTERED_BILL", "AMOUNT_MISMATCH",
              "FUEL_VEHICLE_OTHER_EMPLOYEE", "INSTRUCTION_TEXT", "DATE_MISMATCH", "AMOUNT_OVER_LIMIT"]:
    flagged = ev[ev.ref_reasons.map(lambda x: code_ in x)]
    det.append({"exception (policy reference)": code_, "claims": len(flagged), "sent to review": int((flagged.decision == "review").sum()),
                "same reason code given": int(flagged.reason_codes.str.contains(code_).sum())})
pd.DataFrame(det).set_index("exception (policy reference)")

,claims,sent to review,same reason code given
exception (policy reference),,,
MISSING_IMAGE,3,3,3
UNREADABLE_IMAGE,3,3,2
DUPLICATE_BILL,16,16,15
DUPLICATE_PHOTO,2,2,2
ALTERED_BILL,4,4,3
AMOUNT_MISMATCH,14,14,12
FUEL_VEHICLE_OTHER_EMPLOYEE,3,3,3
INSTRUCTION_TEXT,3,3,3
DATE_MISMATCH,8,8,7


**The 60-claim OCR holdout** (notebook 03) is a later slice enriched with hard cases and partly investigated by hand while building
the parser, so it is reported only as a subset, not as an independent production estimate:

In [19]:
hold = read(ROOT / "reports" / "ocr_benchmark_rows.csv")
hold_ids = set(hold.claim_id)
h = ev[ev.index.isin(hold_ids)]
pd.DataFrame({"claims": [len(h)], "engine approves": [(h.decision == "approve").sum()], "reference approves": [(h.ref_decision == "approve").sum()],
              "unsafe approvals": [((h.decision == "approve") & (h.ref_decision == "review")).sum()],
              "extra reviews": [((h.decision == "review") & (h.ref_decision == "approve")).sum()]}, index=["OCR holdout subset"])

,claims,engine approves,reference approves,unsafe approvals,extra reviews
OCR holdout subset,60,37,46,0,9


## 9 · What the clerks paid that the policy says a person should have stopped

Two months of Nagpur-division claims (train), clerk-approved, where the reference evidence shows an exception. "Paid" assumes approved
means paid (there is no payment record). Over-claims count the excess only; re-claims and colleague bills count the whole claim.

In [20]:
paid = ev[(ev.status == "approved")].join(ref[["clerk_bill_amount"]])
def leak(r):
    reasons = set(r.ref_reasons)
    if reasons & {"DUPLICATE_BILL", "DUPLICATE_PHOTO", "FUEL_VEHICLE_OTHER_EMPLOYEE"}:
        return float(r.amount_claimed), "re-claimed or colleague's bill"
    if reasons & {"AMOUNT_MISMATCH", "ALTERED_BILL"} and float(r.amount_claimed) > float(r.clerk_bill_amount) + 5:
        return float(r.amount_claimed) - float(r.clerk_bill_amount), "claimed more than the bill"
    return 0.0, ""
paid[["leak_rs", "leak_kind"]] = paid.apply(lambda r: pd.Series(leak(r)), axis=1)
leaks = paid[paid.leak_rs > 0]
caught = leaks.decision == "review"
print(f"clerk-approved claims with a policy exception that cost money: {len(leaks)} claims, Rs {leaks.leak_rs.sum():,.0f} "
      f"over two months (~Rs {leaks.leak_rs.sum() / 2:,.0f}/month in this division)")
print(f"the engine would have sent {caught.sum()} of them to review: Rs {leaks[caught].leak_rs.sum():,.0f} of the Rs {leaks.leak_rs.sum():,.0f}")
leaks.groupby("leak_kind").leak_rs.agg(["count", "sum"])

clerk-approved claims with a policy exception that cost money: 25 claims, Rs 14,283 over two months (~Rs 7,142/month in this division)
the engine would have sent 25 of them to review: Rs 14,283 of the Rs 14,283


,count,sum
leak_kind,,
claimed more than the bill,9,4319.8
re-claimed or colleague's bill,16,9963.4


## 10 · Test claims: frozen rules, predictions

The 190 test claims run through the same frozen rules. Fields come from the combined reader (notebook 03). For a claim with no photo,
every receipt field stays blank: nothing is invented and the rep's typed values are **not** copied in as if read from a receipt.

In [21]:
te = decisions[decisions.split == "test"]
sample = read(ROOT / "data" / "raw" / "sample_submission.csv")
F = FIELDS.loc[sample.claim_id]


def short_reason(cid):
    d = decisions.loc[cid]
    if d.decision == "approve":
        return "Approved: amount, bill number, date and vendor read and checked; no duplicate or policy exception" + (
            f" ({d.notes})" if d.notes else "")
    return "Review: " + d.reason


predictions = pd.DataFrame({
    "claim_id": sample.claim_id.values,
    "vendor": F.vendor.values,
    "receipt_date": F.receipt_date.values,
    "receipt_number": F.receipt_number.values,
    "gstin": F.gstin.values,
    "total_amount": F.total_amount.values,
    "decision": [decisions.loc[c, "decision"] for c in sample.claim_id],
    "reason": [short_reason(c) for c in sample.claim_id],
})
assert list(predictions.columns) == list(sample.columns) and len(predictions) == len(sample) == 190
assert predictions.claim_id.is_unique and set(predictions.decision) <= {"approve", "review"}
missing = test[test.photo_available != "True"].claim_id
assert (predictions.set_index("claim_id").loc[missing, ["vendor", "receipt_date", "receipt_number", "gstin", "total_amount"]] == "").all().all()
predictions.to_csv(ROOT / "predictions.csv", index=False)
print(f"predictions.csv: {len(predictions)} rows, approve {(predictions.decision == 'approve').sum()}, review {(predictions.decision == 'review').sum()} "
      f"| rules version {RULES_VERSION}")
te.reason_codes.str.split("; ").explode().loc[lambda s: s != ""].value_counts().to_frame("test claims")

predictions.csv: 190 rows, approve 129, review 61 | rules version 95bef2d00f35


,test claims
reason_codes,
AMOUNT_OVER_LIMIT,16
DUPLICATE_BILL,15
AMOUNT_UNVERIFIED,14
FIELDS_INCOMPLETE,11
DATE_MISMATCH,10
AMOUNT_MISMATCH,10
LODGING_NOT_GST_INVOICE,4
MISSING_IMAGE,3
ALTERED_BILL,3


## 11 · What it saves (assumptions from the policy pack)

Policy §5–6: 2,400 photo claims a month company-wide, Nagpur about a tenth; 3.5 minutes to type and check a claim; two clerks at
Rs 38,000 a month each. Converting that salary to an hourly rate needs an assumption: **160 paid hours per clerk per month**, so
Rs 76,000 / 320 h = **Rs 237.50 per clerk-hour**. Hours freed are capacity, not cash, unless headcount or overtime changes.

The draft proposal (P7) assumes ~90% of claims are auto-approved (everything under Rs 2,000). The engine's **measured** auto-approval
rate is lower, because it also stops duplicates, mismatches and unconfirmed fields.

In [22]:
RATE = 76000 / 320
rows = []
for label, share in [("P7 draft (every claim < Rs 2,000)", 0.90), ("engine on train", (tr.decision == "approve").mean()),
                     ("engine on test", (te.decision == "approve").mean())]:
    for scope, n in [("Nagpur division (~240/month)", 240), ("company (~2,400/month)", 2400)]:
        hours = n * share * 3.5 / 60
        rows.append({"auto-approval rate": label, "scope": scope, "share": share, "claims/month": round(n * share),
                     "clerk hours freed/month": round(hours, 1), "capacity value Rs/month": round(hours * RATE)})
pd.DataFrame(rows).style.format({"share": "{:.0%}"})

,auto-approval rate,scope,share,claims/month,clerk hours freed/month,capacity value Rs/month
0,"P7 draft (every claim < Rs 2,000)",Nagpur division (~240/month),90%,216,12.600000,2992
1,"P7 draft (every claim < Rs 2,000)","company (~2,400/month)",90%,2160,126.000000,29925
2,engine on train,Nagpur division (~240/month),65%,155,9.100000,2150
3,engine on train,"company (~2,400/month)",65%,1552,90.500000,21501
4,engine on test,Nagpur division (~240/month),68%,163,9.500000,2258
5,engine on test,"company (~2,400/month)",68%,1629,95.100000,22575


**The bigger number is leakage, not typing time.** In two months the Nagpur clerks approved claims that the policy says should have
been stopped (section 9). The P7 draft as written (auto-approve everything under Rs 2,000 from the photo) would have paid **all** of
those small ones automatically; the engine flags them for review before payment. Whether they were actually paid, and how much is
avoided, needs payment records and the outcome of human review: approved is only a proxy for paid. Model cost (estimate):
PaddleOCR-VL ran at ~3 s per receipt on a free Kaggle T4, so 2,400 receipts a month is about 2 GPU-hours of inference; retries,
idle time and deployment overhead are not included.

## 12 · Save

In [23]:
out = decisions.join(claims.set_index("claim_id")[["category", "amount_claimed", "emp_id"]])
out = out.join(ref[["ref_decision", "ref_reasons"]].assign(ref_reasons=lambda d: d.ref_reasons.map("; ".join)))
out["rules_version"] = RULES_VERSION
out.drop(columns="decision_ms").reset_index().to_csv(ROOT / "reports" / "decisions.csv", index=False)   # timings vary per run; median is in the summary
summary = pd.DataFrame([
    {"metric": "auto-approval coverage (train)", "value": f"{(tr.decision == 'approve').mean():.1%}"},
    {"metric": "auto-approval coverage (test)", "value": f"{(te.decision == 'approve').mean():.1%}"},
    {"metric": "unsafe auto-approvals (train, vs policy reference)", "value": f"{len(unsafe)} of {n_review_worthy} review-worthy"},
    {"metric": "extra reviews (train)", "value": f"{len(extra)} of {len(ev)}"},
    {"metric": "agreement with clerk decisions (train)", "value": f"{agree:.1%}"},
    {"metric": "known cases sent to review", "value": f"{len(reg)} of {len(reg)}"},
    {"metric": "decision latency, median", "value": f"{decisions.decision_ms.median():.1f} ms (+ ~3 s reading on a T4)"},
    {"metric": "rules version", "value": RULES_VERSION},
])
summary.to_csv(ROOT / "reports" / "decision_summary.csv", index=False)
summary

,metric,value
0,auto-approval coverage (train),64.7%
1,auto-approval coverage (test),67.9%
2,"unsafe auto-approvals (train, vs policy refere...",0 of 60 review-worthy
3,extra reviews (train),40 of 283
4,agreement with clerk decisions (train),67.5%
5,known cases sent to review,26 of 26
6,"decision latency, median",0.8 ms (+ ~3 s reading on a T4)
7,rules version,95bef2d00f35
